# Chatbot UG - Admisión y Nivelación


**Ejecuta las celdas en orden.**

In [1]:
!pip -q install gradio scikit-learn pandas nltk

In [2]:
import json, re, unicodedata, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.stem.snowball import SnowballStemmer
import gradio as gr

In [3]:
# Cargar la base incluida en el proyecto
RUTA = "/content/base_conocimiento.json"
try:
    registros = json.load(open(RUTA, encoding="utf-8"))
except FileNotFoundError:
    from google.colab import files
    print("Sube data/base_conocimiento.json")
    files.upload()
    registros = json.load(open("base_conocimiento.json", encoding="utf-8"))
print("Registros:", len(registros))

Sube data/base_conocimiento.json


Saving base_conocimiento.json to base_conocimiento.json
Registros: 87


In [4]:
def normalizar(texto):
    texto=str(texto).lower().strip()
    texto="".join(c for c in unicodedata.normalize("NFD",texto) if unicodedata.category(c)!="Mn")
    texto=re.sub(r"[^a-z0-9\s]"," ",texto)
    return re.sub(r"\s+"," ",texto).strip()

# Lista propia de stopwords en español (no depende de descargas externas).
STOPWORDS_ES={"a","al","algo","algunas","algunos","ante","antes","como","con","contra","cual","cuando","de","del","desde","donde","durante","e","el","ella","ellas","ellos","en","entre","era","es","esa","esas","ese","eso","esos","esta","estaba","estan","estar","este","esto","estos","estoy","fue","fueron","ha","habia","han","has","hasta","hay","he","la","las","le","les","lo","los","mas","me","mi","mientras","mis","mucho","muchos","muy","nada","ni","no","nos","nosotras","nosotros","nuestra","nuestro","o","otra","otro","para","pero","poco","por","porque","que","quien","quienes","se","sea","sera","si","sido","siendo","sin","sobre","sois","somos","son","soy","su","sus","tambien","tanto","te","ti","tiene","tienen","tienes","todo","todos","tu","tus","un","una","uno","unos","vosotras","vosotros","vuestra","vuestro","y","ya","yo"}
STEMMER=SnowballStemmer("spanish")

def preprocesar(texto):
    tokens=normalizar(texto).split()
    tokens=[t for t in tokens if t not in STOPWORDS_ES and len(t)>1]
    tokens=[STEMMER.stem(t) for t in tokens]
    return " ".join(tokens)

def texto_registro(r):
    return preprocesar(" ".join(str(r.get(c,"")) for c in ["tipo","titulo","carrera","modalidad","descripcion","detalle"]))

textos=[texto_registro(r) for r in registros]
vectorizador=TfidfVectorizer(ngram_range=(1,2))
X=vectorizador.fit_transform(textos)
print(X.shape)

(87, 1115)


In [5]:
UMBRAL=0.18

# Reglas por expresión regular para saludo y despedida (se evalúan antes
# que la similitud coseno, igual que la regla de cédula/pasaporte).
PATRON_SALUDO=re.compile(r'\b(hola+|buenos dias|buenas tardes|buenas noches|buenas|que tal|hey|saludos|hi|hello)\b')
PATRON_DESPEDIDA=re.compile(r'\b(adios|chao|hasta luego|nos vemos|gracias|muchas gracias|eso es todo|bye)\b')
SALUDO_RESPUESTA=("¡Hola! Soy el asistente virtual de Admisión y Nivelación de la UG. \n"
    "Puedo ayudarte con: carreras, requisitos de admisión, cronograma por cédula, "
    "matrícula (nivelación y ordinaria, incluido cuarto semestre y otros niveles), "
    "homologación, becas y trámites del SIUG.\n¿Sobre qué te gustaría preguntar?")
DESPEDIDA_RESPUESTA=("¡Con gusto! Si tienes otra consulta sobre admisión, nivelación o vida "
    "universitaria, aquí estaré. ¡Éxitos! ")
FALLBACK_RESPUESTA=("No encontré una respuesta suficientemente clara para tu consulta.\n"
    "Intenta reformularla mencionando, por ejemplo: una carrera, 'requisitos de admisión', "
    "'cronograma', 'nivelación', 'matrícula ordinaria', 'homologación' o 'becas'.")

def responder(consulta):
    if not str(consulta).strip():
        return "Escribe una consulta. Por ejemplo: '¿Qué modalidad tiene Medicina?'"
    qn=normalizar(consulta)
    if PATRON_SALUDO.search(qn):
        return SALUDO_RESPUESTA
    if PATRON_DESPEDIDA.search(qn):
        return DESPEDIDA_RESPUESTA
    m=re.search(r'(?:termina en|ultimo digito(?: es)?|digito)\s*(\d)',qn)
    nums=re.findall(r'\d{10}',qn)
    ultimo=nums[0][-1] if nums else (m.group(1) if m else None)
    if ultimo and any(x in qn for x in ["cedula","pasaporte","digito","inscrib","postul"]):
        mapa={"1":"lunes 13-jul-26","2":"lunes 13-jul-26","3":"martes 14-jul-26","4":"martes 14-jul-26","5":"miércoles 15-jul-26","6":"miércoles 15-jul-26","7":"jueves 16-jul-26","8":"jueves 16-jul-26","9":"viernes 17-jul-26","0":"viernes 17-jul-26"}
        return f"Tu documento termina en {ultimo}. Puedes ingresar desde las 09:00 am el {mapa[ultimo]}."
    sims=cosine_similarity(vectorizador.transform([preprocesar(consulta)]),X).flatten()
    idx=int(sims.argmax()); score=float(sims[idx]); r=registros[idx]
    if score<UMBRAL:
        return FALLBACK_RESPUESTA
    partes=[]
    if r.get("carrera"): partes.append(f"**Carrera:** {r['carrera']}")
    if r.get("modalidad"): partes.append(f"**Modalidad:** {r['modalidad']}")
    if r.get("titulo"): partes.append(f"**Tema:** {r['titulo']}")
    if r.get("descripcion"): partes.append(r["descripcion"])
    if r.get("detalle"): partes.append(r["detalle"])
    partes.append(f"_Similitud: {score:.4f}_")
    return "\n\n".join(partes)

In [ ]:
# ============================================================
# INTERFAZ GRADIO CON PALETA INSTITUCIONAL UG
# Azul marino, azul universitario, blanco y detalles dorados
# ============================================================

CSS_UG = """
:root {
    --ug-azul-marino: #082B5C;
    --ug-azul: #0D47A1;
    --ug-azul-claro: #EAF2FF;
    --ug-dorado: #D5A300;
    --ug-blanco: #FFFFFF;
    --ug-texto: #162033;
    --ug-borde: #B7C9E5;
}

/* Contenedor general */
.gradio-container {
    max-width: 1040px !important;
    margin: 0 auto !important;
    padding: 28px 32px 40px !important;
    background:
        linear-gradient(180deg, #FFFFFF 0%, #F7FAFF 100%) !important;
    color: var(--ug-texto) !important;
    font-family: Arial, Helvetica, sans-serif !important;
}

/* Título principal */
.gradio-container h1 {
    position: relative !important;
    margin-bottom: 42px !important;
    color: var(--ug-azul-marino) !important;
    font-family: Georgia, "Times New Roman", serif !important;
    font-size: clamp(2rem, 4vw, 3rem) !important;
    font-weight: 700 !important;
    line-height: 1.15 !important;
    text-align: center !important;
    letter-spacing: -0.02em !important;
}

/* Línea dorada ornamental bajo el título */
.gradio-container h1::after {
    content: "" !important;
    display: block !important;
    width: min(390px, 75%) !important;
    height: 16px !important;
    margin: 14px auto 0 !important;
    background:
        linear-gradient(var(--ug-dorado), var(--ug-dorado))
        left center / 46% 2px no-repeat,
        radial-gradient(circle, var(--ug-dorado) 0 5px, transparent 6px)
        center / 14px 14px no-repeat,
        linear-gradient(var(--ug-dorado), var(--ug-dorado))
        right center / 46% 2px no-repeat !important;
}

/* Tarjetas y bloques */
.gradio-container .block,
.gradio-container .form {
    border-color: var(--ug-borde) !important;
    border-radius: 12px !important;
}

.gradio-container .form {
    background: rgba(255, 255, 255, 0.92) !important;
    box-shadow: 0 8px 24px rgba(8, 43, 92, 0.06) !important;
}

/* Etiquetas */
.gradio-container label span,
.gradio-container .label-wrap span {
    color: var(--ug-azul-marino) !important;
    font-weight: 700 !important;
    font-size: 1rem !important;
}

/* Caja de pregunta */
.gradio-container textarea,
.gradio-container input[type="text"] {
    color: var(--ug-texto) !important;
    background: var(--ug-blanco) !important;
    border: 2px solid var(--ug-azul) !important;
    border-radius: 10px !important;
    box-shadow: 0 0 0 3px rgba(13, 71, 161, 0.05) !important;
    font-size: 1rem !important;
}

.gradio-container textarea:focus,
.gradio-container input[type="text"]:focus {
    border-color: var(--ug-dorado) !important;
    box-shadow: 0 0 0 4px rgba(213, 163, 0, 0.18) !important;
}

/* Botón principal: Submit */
.gradio-container button.primary {
    color: var(--ug-blanco) !important;
    background: linear-gradient(
        135deg,
        var(--ug-azul-marino) 0%,
        var(--ug-azul) 100%
    ) !important;
    border: 1px solid var(--ug-azul-marino) !important;
    border-radius: 10px !important;
    font-weight: 700 !important;
    box-shadow: 0 8px 18px rgba(8, 43, 92, 0.18) !important;
    transition: transform 0.18s ease, box-shadow 0.18s ease !important;
}

.gradio-container button.primary:hover {
    transform: translateY(-1px) !important;
    box-shadow: 0 10px 22px rgba(8, 43, 92, 0.25) !important;
}

/* Botones secundarios: Clear y Flag */
.gradio-container button.secondary {
    color: var(--ug-azul-marino) !important;
    background: var(--ug-blanco) !important;
    border: 2px solid var(--ug-azul) !important;
    border-radius: 10px !important;
    font-weight: 700 !important;
}

.gradio-container button.secondary:hover {
    color: var(--ug-blanco) !important;
    background: var(--ug-azul-marino) !important;
    border-color: var(--ug-azul-marino) !important;
}

/* Respuesta del chatbot */
#respuesta-ug {
    padding: 10px 4px 6px !important;
    color: var(--ug-texto) !important;
    font-size: 1rem !important;
    line-height: 1.75 !important;
}

#respuesta-ug strong {
    color: var(--ug-azul-marino) !important;
}

/* Zona de ejemplos */
.gradio-container .examples {
    margin-top: 10px !important;
}

.gradio-container .examples > .label,
.gradio-container .examples .label {
    color: var(--ug-azul-marino) !important;
    font-weight: 700 !important;
}

/* Botones de preguntas de ejemplo */
.gradio-container .examples button,
.gradio-container button.example {
    color: var(--ug-azul-marino) !important;
    background: var(--ug-blanco) !important;
    border: 1.5px solid #6E96CE !important;
    border-radius: 9px !important;
    font-weight: 500 !important;
    transition: background 0.18s ease, border-color 0.18s ease !important;
}

.gradio-container .examples button:hover,
.gradio-container button.example:hover {
    color: var(--ug-azul-marino) !important;
    background: var(--ug-azul-claro) !important;
    border-color: var(--ug-dorado) !important;
}

/* Enlaces y pequeños acentos */
.gradio-container a {
    color: var(--ug-azul) !important;
}

/* Ajuste para pantallas pequeñas */
@media (max-width: 700px) {
    .gradio-container {
        padding: 18px 14px 30px !important;
    }

    .gradio-container h1 {
        margin-bottom: 30px !important;
        font-size: 2rem !important;
    }
}
"""

tema_ug = gr.themes.Soft(
    primary_hue=gr.themes.colors.blue,
    secondary_hue=gr.themes.colors.amber,
    neutral_hue=gr.themes.colors.slate
)

demo = gr.Interface(
    fn=responder,
    inputs=gr.Textbox(
        lines=3,
        label="Pregunta",
        placeholder="Escribe aquí tu consulta..."
    ),
    outputs=gr.Markdown(
        label="Respuesta",
        elem_id="respuesta-ug"
    ),
    title="Chatbot UG - Admisión y Nivelación",
    description=(
        "Asistente virtual para consultas sobre carreras, admisión, "
        "nivelación, matrícula, homologación, becas y trámites del SIUG."
    ),
    examples=[
        ["Hola"],
        ["¿Qué modalidad tiene Medicina?"],
        ["¿Qué documentos necesito para nivelación?"],
        ["Mi cédula termina en 8, ¿qué día postulo?"],
        ["¿Cuál es el promedio mínimo para aprobar nivelación?"],
        ["Estoy en cuarto semestre, ¿cómo me matriculo?"],
        ["Gracias por la ayuda"]
    ],
    theme=tema_ug,
    css=CSS_UG
)

demo.launch(share=True, debug=True)

/usr/local/lib/python3.12/dist-packages/gradio/interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  super().__init__(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ce2bd196b18f681c14.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
